# Lantern — Hybrid Recommender: Training & Pre-computation
**MINE4201-01 · Taller 2 · Semester 2026-1**

This notebook:
1. Downloads / loads the Yelp Open Dataset (Philadelphia subset)
2. Trains a Collaborative Filtering model via Alternating Least Squares (ALS)
3. Computes a context-aware re-ranking component
4. Pre-computes top-50 recommendations per user → `data/top_n.parquet`
5. Pre-computes signal breakdown per (user, business) pair → `data/explanations.parquet`
6. Evaluates offline: Recall@K, NDCG@K, MAP → `data/eval.json`

In [ ]:
import json, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
import implicit

warnings.filterwarnings('ignore')
DATA_DIR = Path('..') / 'data'
DATA_DIR.mkdir(exist_ok=True)

print(f'implicit version: {implicit.__version__}')
print(f'numpy version: {np.__version__}')

## 1. Load Yelp Data
Place the Yelp Open Dataset files in `backend/data/yelp_dataset/`:
- `yelp_academic_dataset_review.json` 
- `yelp_academic_dataset_business.json`

We filter to Philadelphia businesses only.

In [ ]:
YELP_DIR = DATA_DIR / 'yelp_dataset'
USE_SAMPLE = not (YELP_DIR / 'yelp_academic_dataset_business.json').exists()

if USE_SAMPLE:
    print('⚠️  Yelp dataset not found — using synthetic sample for development')
    # Generate synthetic data that mimics Yelp structure
    np.random.seed(42)
    N_BIZ = 200
    N_USERS = 500
    N_REVIEWS = 5000

    biz_ids = [f'biz_{i:04d}' for i in range(N_BIZ)]
    user_ids = [f'user_{i:04d}' for i in range(N_USERS)]
    categories_pool = ['Italian', 'Vietnamese', 'Brunch', 'Cocktail Bars',
                       'Mediterranean', 'Thai', 'Mexican', 'Specialty Coffee', 'Vegan']
    neighborhoods_pool = ['Bella Vista', 'Fishtown', 'Old City', 'Center City',
                          'Fairmount', 'South Philly', 'Kensington', 'Point Breeze']

    businesses_df = pd.DataFrame({
        'business_id': biz_ids,
        'name': [f'Restaurant {i}' for i in range(N_BIZ)],
        'city': 'Philadelphia',
        'state': 'PA',
        'categories': [np.random.choice(categories_pool) for _ in range(N_BIZ)],
        'neighborhood': [np.random.choice(neighborhoods_pool) for _ in range(N_BIZ)],
        'stars': np.random.uniform(3.0, 5.0, N_BIZ).round(1),
        'review_count': np.random.randint(10, 2000, N_BIZ),
        'is_open': 1,
    })

    # Simulate implicit feedback: reviews with ratings >= 3 = positive interaction
    reviews_df = pd.DataFrame({
        'review_id': [f'rev_{i}' for i in range(N_REVIEWS)],
        'user_id': np.random.choice(user_ids, N_REVIEWS),
        'business_id': np.random.choice(biz_ids, N_REVIEWS),
        'stars': np.random.choice([3, 4, 5], N_REVIEWS, p=[0.2, 0.4, 0.4]),
        'date': pd.date_range('2020-01-01', periods=N_REVIEWS, freq='2h'),
    })
    # Also add our mock frontend businesses as known entities
    mock_biz_ids = ['otello','miss-rachels','suraya','high-street','double-knot',
                    'elixr','pho-79','a-mano','kalaya']
    mock_rows = pd.DataFrame({
        'business_id': mock_biz_ids,
        'name': ['Otello','Miss Rachel\'s Pantry','Suraya','High Street','Double Knot',
                 'ReAnimator','Phở 79','A Mano','Kalaya'],
        'city': 'Philadelphia', 'state': 'PA',
        'categories': ['Italian','Vegan','Mediterranean','Brunch','Cocktail Bars',
                       'Specialty Coffee','Vietnamese','Italian','Thai'],
        'neighborhood': ['Bella Vista','Point Breeze','Fishtown','Old City','Center City',
                         'Kensington','South Philly','Fairmount','Fishtown'],
        'stars': [4.6,4.5,4.7,4.4,4.5,4.6,4.3,4.5,4.6],
        'review_count': [847,412,1184,936,612,528,1043,287,723],
        'is_open': 1,
    })
    businesses_df = pd.concat([businesses_df, mock_rows], ignore_index=True)

    # Add reviews for our mock user (camila) and mock businesses
    camila_reviews = pd.DataFrame({
        'review_id': [f'camila_rev_{i}' for i in range(len(mock_biz_ids[:6]))],
        'user_id': 'camila',
        'business_id': mock_biz_ids[:6],
        'stars': [5, 4, 5, 4, 5, 4],
        'date': pd.date_range('2024-01-01', periods=6, freq='30D'),
    })
    reviews_df = pd.concat([reviews_df, camila_reviews], ignore_index=True)

else:
    print('📂 Loading Yelp Open Dataset...')
    chunks = []
    with open(YELP_DIR / 'yelp_academic_dataset_business.json', 'r', encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d.get('city') == 'Philadelphia' and d.get('is_open', 0) == 1:
                chunks.append(d)
    businesses_df = pd.DataFrame(chunks)
    businesses_df['neighborhood'] = businesses_df.get('attributes', {}) 
    print(f'  Philadelphia businesses: {len(businesses_df):,}')

    # Load reviews — stream to limit memory
    philly_biz_ids = set(businesses_df['business_id'])
    rev_chunks = []
    with open(YELP_DIR / 'yelp_academic_dataset_review.json', 'r', encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d['business_id'] in philly_biz_ids:
                rev_chunks.append({
                    'review_id': d['review_id'],
                    'user_id': d['user_id'],
                    'business_id': d['business_id'],
                    'stars': d['stars'],
                    'date': d['date'],
                })
    reviews_df = pd.DataFrame(rev_chunks)
    print(f'  Philadelphia reviews: {len(reviews_df):,}')

print(f'Businesses: {len(businesses_df):,} | Reviews: {len(reviews_df):,}')

## 2. Preprocess — Build Interaction Matrix

In [ ]:
# Keep only positive interactions (stars >= 3)
pos = reviews_df[reviews_df['stars'] >= 3].copy()

# Min activity filter: user >= 3 reviews, business >= 5 reviews
user_counts = pos['user_id'].value_counts()
biz_counts  = pos['business_id'].value_counts()
pos = pos[
    pos['user_id'].isin(user_counts[user_counts >= 3].index) &
    pos['business_id'].isin(biz_counts[biz_counts >= 5].index)
]

# Deduplicate (keep highest rating per user-biz pair)
pos = pos.sort_values('stars', ascending=False).drop_duplicates(['user_id','business_id'])

# Encode IDs
user_enc = LabelEncoder()
biz_enc  = LabelEncoder()
pos['u_idx'] = user_enc.fit_transform(pos['user_id'])
pos['b_idx'] = biz_enc.fit_transform(pos['business_id'])

# Implicit confidence: c_ui = 1 + alpha * r_ui  (alpha=10)
alpha = 10
pos['confidence'] = 1 + alpha * pos['stars']

N_USERS = pos['u_idx'].max() + 1
N_ITEMS = pos['b_idx'].max() + 1

# Build sparse item-user matrix (implicit expects items as rows)
user_item = sp.csr_matrix(
    (pos['confidence'], (pos['u_idx'], pos['b_idx'])),
    shape=(N_USERS, N_ITEMS)
)
item_user = user_item.T.tocsr()

print(f'Users: {N_USERS:,} | Items: {N_ITEMS:,} | Interactions: {pos.shape[0]:,}')
print(f'Matrix density: {pos.shape[0] / (N_USERS * N_ITEMS):.4%}')

## 3. Train / Test Split

In [ ]:
# Leave-one-out: for each user, hold out the most recent interaction
pos_sorted = pos.sort_values('date') if 'date' in pos.columns else pos
test_set = pos_sorted.groupby('u_idx').tail(1)
train_set = pos_sorted.drop(test_set.index)

train_mat = sp.csr_matrix(
    (train_set['confidence'], (train_set['u_idx'], train_set['b_idx'])),
    shape=(N_USERS, N_ITEMS)
).T.tocsr()

print(f'Train: {len(train_set):,} | Test: {len(test_set):,}')

## 4. Train ALS Model

In [ ]:
model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.01,
    alpha=alpha,
    use_gpu=False,
    random_state=42,
)

model.fit(train_mat)
print('✅ ALS model trained')
print(f'   factors={model.factors}, iterations={model.iterations}')

## 5. Context-Aware Re-ranker
Simple rule-based context signals on top of CF scores:
- `time_of_day`: evening → boost cozy/special, morning → boost coffee/brunch
- `popularity`: log(review_count) normalized
- `category_affinity`: from user taste profile

In [ ]:
# Build a category-to-context mapping
CATEGORY_CONTEXT = {
    'Italian': ['cozy','special occasion'],
    'Vietnamese': ['cheap eats','asian'],
    'Brunch': ['cozy','brunch'],
    'Cocktail Bars': ['lively','cocktails'],
    'Mediterranean': ['lively','special occasion'],
    'Thai': ['lively','asian'],
    'Mexican': ['lively','mexican'],
    'Specialty Coffee': ['cheap eats','coffee'],
    'Vegan': ['cozy','cheap eats'],
    'Wine Bars': ['lively','wine'],
}

# Merge businesses with their category context
biz_meta = businesses_df[['business_id','categories','stars','review_count']].copy()
biz_meta['log_pop'] = np.log1p(biz_meta['review_count'].fillna(1))
biz_meta['pop_norm'] = (biz_meta['log_pop'] - biz_meta['log_pop'].min()) / \
                        (biz_meta['log_pop'].max() - biz_meta['log_pop'].min() + 1e-9)

# Index biz_meta by business_id for fast lookup
biz_meta_idx = biz_meta.set_index('business_id')

def context_score(business_id: str, hour: int = 20) -> float:
    """Returns a 0-1 context bonus based on time of day."""
    if business_id not in biz_meta_idx.index:
        return 0.0
    row = biz_meta_idx.loc[business_id]
    cat = str(row.get('categories', ''))
    score = 0.0
    # Evening (17-23) → boost cozy/special places
    if 17 <= hour <= 23:
        if any(c in cat for c in ['Italian','Cocktail','Wine','Mediterranean','Thai']):
            score += 0.3
    # Morning (6-11) → boost coffee/brunch
    elif 6 <= hour <= 11:
        if any(c in cat for c in ['Coffee','Brunch']):
            score += 0.3
    return min(score, 1.0)

print('✅ Context scorer ready')

## 6. Pre-compute Top-50 per User

In [ ]:
TOP_N = 50
W_CF  = 0.60  # Collaborative filtering weight
W_CTX = 0.25  # Context weight
W_POP = 0.15  # Popularity weight
HOUR  = 20    # Default: evening

rows = []

# Get CF scores for all users
user_ids_enc = list(range(N_USERS))

# Use implicit's recommend() for efficiency
user_item_train = train_mat.T.tocsr()  # back to user×item

for u_idx in user_ids_enc:
    # CF: top-200 candidates from ALS
    try:
        biz_indices, cf_scores = model.recommend(
            u_idx, user_item_train[u_idx], N=200, filter_already_liked_items=True
        )
    except Exception:
        continue

    if len(biz_indices) == 0:
        continue

    # Normalize CF scores to [0,1]
    cf_min, cf_max = cf_scores.min(), cf_scores.max()
    cf_norm = (cf_scores - cf_min) / (cf_max - cf_min + 1e-9)

    user_id_str = user_enc.inverse_transform([u_idx])[0]

    for rank, (b_idx, cf_n) in enumerate(zip(biz_indices, cf_norm)):
        biz_id = biz_enc.inverse_transform([b_idx])[0]

        # Context score
        ctx_n = context_score(biz_id, hour=HOUR)

        # Popularity score
        pop_n = float(biz_meta_idx.loc[biz_id, 'pop_norm']) \
                if biz_id in biz_meta_idx.index else 0.0

        # Hybrid score
        hybrid = W_CF * cf_n + W_CTX * ctx_n + W_POP * pop_n

        # Signal contribution as % of total
        cf_contrib  = int(round(W_CF  * cf_n  / (hybrid + 1e-9) * 100))
        ctx_contrib = int(round(W_CTX * ctx_n / (hybrid + 1e-9) * 100))
        pop_contrib = max(0, 100 - cf_contrib - ctx_contrib)

        rows.append({
            'user_id': user_id_str,
            'business_id': biz_id,
            'rank': rank + 1,
            'score': round(float(hybrid), 6),
            'cf': cf_contrib,
            'ctx': ctx_contrib,
            'pop': pop_contrib,
        })

        if rank >= TOP_N - 1:
            break

top_n_df = pd.DataFrame(rows)
top_n_df.to_parquet(DATA_DIR / 'top_n.parquet', index=False)
print(f'✅ top_n.parquet saved — {len(top_n_df):,} rows, {top_n_df["user_id"].nunique():,} users')

## 7. Pre-compute Explanations per (user, business)

In [ ]:
# explanations.parquet — same as top_n but keyed for fast (user_id, business_id) lookup
explanations_df = top_n_df[['user_id','business_id','score','cf','ctx','pop']].copy()
explanations_df['match'] = (explanations_df['score'] * 100).clip(0, 100).round().astype(int)
explanations_df.to_parquet(DATA_DIR / 'explanations.parquet', index=False)
print(f'✅ explanations.parquet saved — {len(explanations_df):,} rows')

## 8. Offline Evaluation: Recall@K, NDCG@K, MAP

In [ ]:
K_VALUES = [5, 10, 20]
results = {}

test_user_items = test_set.groupby('u_idx')['b_idx'].apply(list)

for K in K_VALUES:
    recalls, ndcgs, aps = [], [], []

    for u_idx, held_out in test_user_items.items():
        try:
            pred_indices, _ = model.recommend(
                u_idx, user_item_train[u_idx], N=K, filter_already_liked_items=True
            )
        except Exception:
            continue

        hits = set(pred_indices) & set(held_out)
        recall = len(hits) / len(held_out) if held_out else 0
        recalls.append(recall)

        # NDCG@K
        relevance = np.zeros(K)
        for i, idx in enumerate(pred_indices[:K]):
            if idx in held_out:
                relevance[i] = 1
        ndcg = ndcg_score([np.ones(K)], [relevance]) if relevance.sum() > 0 else 0
        ndcgs.append(ndcg)

        # Average Precision@K
        ap, n_relevant = 0.0, 0
        for i, idx in enumerate(pred_indices[:K]):
            if idx in held_out:
                n_relevant += 1
                ap += n_relevant / (i + 1)
        aps.append(ap / max(len(held_out), 1))

    results[f'Recall@{K}'] = round(float(np.mean(recalls)), 4)
    results[f'NDCG@{K}']   = round(float(np.mean(ndcgs)), 4)
    results[f'MAP@{K}']    = round(float(np.mean(aps)), 4)

results['model'] = 'ALS'
results['factors'] = model.factors
results['iterations'] = model.iterations
results['W_CF'] = W_CF
results['W_CTX'] = W_CTX
results['W_POP'] = W_POP
results['n_users'] = int(N_USERS)
results['n_items'] = int(N_ITEMS)
results['n_interactions'] = int(pos.shape[0])

with open(DATA_DIR / 'eval.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n📊 Offline Evaluation Results')
print('=' * 40)
for k, v in results.items():
    if isinstance(v, float):
        print(f'  {k:<15} {v:.4f}')
    elif isinstance(v, (int, str)):
        print(f'  {k:<15} {v}')

print('\n✅ eval.json saved')

## 9. Connect Mock Businesses to Real Model
Map the frontend's 9 hardcoded business IDs to real ALS scores for user 'camila'.

In [ ]:
MOCK_BIZ_IDS = [
    'otello','miss-rachels','suraya','high-street','double-knot',
    'elixr','pho-79','a-mano','kalaya'
]

# Check which mock businesses are in the encoded set
known_in_model = [b for b in MOCK_BIZ_IDS if b in biz_enc.classes_]
print(f'Mock businesses in model: {len(known_in_model)}/{len(MOCK_BIZ_IDS)}')
for b in MOCK_BIZ_IDS:
    status = '✅' if b in known_in_model else '⚠️ not in model (will use mock values)'
    print(f'  {b}: {status}')

# Export which IDs are real vs mock-only (used by the API to decide lookup strategy)
id_map = {'real': known_in_model, 'mock_only': [b for b in MOCK_BIZ_IDS if b not in known_in_model]}
with open(DATA_DIR / 'id_map.json', 'w') as f:
    json.dump(id_map, f, indent=2)
print('\n✅ id_map.json saved')

## Done!
Artifacts generated:
- `data/top_n.parquet` — top-50 recommendations per user with hybrid scores
- `data/explanations.parquet` — signal breakdown (cf/ctx/pop) per (user, business)
- `data/eval.json` — offline metrics: Recall@K, NDCG@K, MAP@K
- `data/id_map.json` — which mock business IDs are in the trained model

The FastAPI server loads these on startup and serves them as pre-computed lookups.